# Phase 17 — Final Deployment

**Objective:** Package the FF++-adapted Xception dual-stream model into a production-ready Streamlit web application and deploy it to HuggingFace Spaces via GitHub. This is the deliverable phase — a publicly accessible explainable deepfake detector that shows Grad-CAM heatmaps and DCT frequency maps alongside every prediction.

**What this phase produces:**
- `app.py` — the complete Streamlit application
- `requirements.txt`, `Dockerfile`, `.streamlit/config.toml` — deployment config
- `README.md` — GitHub repository documentation
- All files packaged as a zip ready to push to GitHub

## Why Phase 17 Exists

Phase 12 deployed a Streamlit app using the **EfficientNetB0 dual-stream** model trained only on CIFAKE. After Phases 13–15, the final model is fundamentally different:

| | Phase 12 (old) | Phase 17 (this) |
|---|---|---|
| Backbone | EfficientNetB0 | **Xception** |
| Training data | CIFAKE only | CIFAKE + **FF++ C23** |
| Face-swap detection | ✗ | ✓ |
| Zero-shot tested | ✗ | ✓ ArtiFact 25 generators |
| Explainability | Basic | Grad-CAM + DCT map side-by-side |

**Deployment architecture:**
```
Kaggle (training) → ffpp_adapted_final.keras
         ↓ upload
HuggingFace Hub  (model weights — free, versioned)
         ↓ hf_hub_download at runtime
HuggingFace Spaces (Docker + Streamlit — live URL, free)
         ↑ git push
GitHub repo      (source code — version controlled)
```

The model weights are **not** in the GitHub repo (too large). They live on HuggingFace Hub and are pulled automatically when the app starts.

## Step 1 — Write All Deployment Files

In [ ]:
import os

# All files will be written to /kaggle/working/deepfake-detector/
DEPLOY_DIR = "/kaggle/working/deepfake-detector"
os.makedirs(f"{DEPLOY_DIR}/.streamlit", exist_ok=True)
print("Deploy directory ready:", DEPLOY_DIR)

### app.py — The Streamlit Application

The entire app in one file. Key sections:
- **Model loading** — downloads from HuggingFace Hub and caches with `@st.cache_resource`
- **Preprocessing** — RGB normalisation + log-DCT frequency map (identical to training pipeline)
- **Inference** — dual-stream `model.predict([rgb, dct])`
- **Grad-CAM** — gradient tape on last Xception conv layer, overlaid on original image
- **UI** — verdict card, probability bar, 3-column explainability panel, sidebar controls

In [ ]:
APP_CODE = '''
"""
Phase 17 — Final Deployment
Beyond Binary Detection: XAI Deepfake Localization via Frequency-Aware Detection
Streamlit app — uses the FF++-adapted Xception dual-stream model.
"""

import os
import cv2
import numpy as np
import streamlit as st
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
from huggingface_hub import hf_hub_download

# ── Page config ───────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Deepfake Detector",
    page_icon="🔍",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── Styling ───────────────────────────────────────────────────────────────────
st.markdown("""
<style>
    .main { background-color: #0f1117; }
    .verdict-real {
        background: #1a3a1a; border: 2px solid #2ecc71;
        border-radius: 10px; padding: 16px; text-align: center;
    }
    .verdict-fake {
        background: #3a1a1a; border: 2px solid #e74c3c;
        border-radius: 10px; padding: 16px; text-align: center;
    }
    .metric-box {
        background: #1e2130; border-radius: 8px;
        padding: 12px; text-align: center; margin: 4px;
    }
    .section-header {
        font-size: 1.1rem; font-weight: 600;
        color: #a0aec0; margin-bottom: 8px;
    }
</style>
""", unsafe_allow_html=True)

# ── Constants ─────────────────────────────────────────────────────────────────
IMG_SIZE       = (299, 299)
CONFIDENCE_LOW = 0.40   # below this → likely real but uncertain
CONFIDENCE_HIGH= 0.60   # above this → likely fake but uncertain
HF_REPO_ID     = "YOUR_HF_USERNAME/deepfake-detector-model"   # ← update after upload
MODEL_FILENAME  = "ffpp_adapted_final.keras"

GENERATOR_INFO = {
    "StyleGAN / StyleGAN2 / StyleGAN3": "Face generation via progressive style injection",
    "Stable Diffusion / Latent Diffusion": "Text-to-image diffusion model",
    "DDPM / Denoising Diffusion GAN": "Noise-based generative diffusion",
    "GANFormer / ProGAN / BigGAN": "Adversarial training on large image distributions",
    "Face2Face / NeuralTextures": "Expression transfer or neural texture rendering",
    "Deepfakes / FaceSwap / FaceShifter": "Autoencoder or 3DMM face identity swap",
}

# ── Model loading ─────────────────────────────────────────────────────────────
@st.cache_resource(show_spinner=False)
def load_model():
    """Download model from HuggingFace Hub and cache it."""
    with st.spinner("Loading model (first run may take ~30 seconds)…"):
        try:
            path = hf_hub_download(repo_id=HF_REPO_ID, filename=MODEL_FILENAME)
        except Exception:
            # Fallback: look for local model (useful during dev)
            path = MODEL_FILENAME
            if not os.path.exists(path):
                st.error(
                    f"Model not found. Upload `{MODEL_FILENAME}` to "
                    f"HuggingFace repo `{HF_REPO_ID}` or place it locally."
                )
                st.stop()
        return tf.keras.models.load_model(path)

# ── Preprocessing ─────────────────────────────────────────────────────────────
def preprocess_rgb(img_array):
    """Resize and normalise to [0,1] for the RGB stream."""
    resized = cv2.resize(img_array, IMG_SIZE).astype(np.float32) / 255.0
    return resized

def preprocess_dct(img_array):
    """Compute log-DCT frequency map for the frequency stream."""
    img_u8 = img_array.astype(np.uint8)
    gray   = cv2.cvtColor(img_u8, cv2.COLOR_RGB2GRAY)
    gray   = cv2.resize(gray, IMG_SIZE).astype(np.float32)
    dct    = cv2.dct(gray)
    dct    = np.log(np.abs(dct) + 1)
    dct    = cv2.normalize(dct, None, 0, 1, cv2.NORM_MINMAX)
    return np.stack([dct] * 3, axis=-1)

# ── Grad-CAM ──────────────────────────────────────────────────────────────────
def make_gradcam(model, rgb_input, dct_input, last_conv_layer="block14_sepconv2_act"):
    """
    Generate Grad-CAM heatmap on the RGB stream's last conv layer.
    Falls back gracefully if the layer name doesn't match.
    """
    try:
        grad_model = tf.keras.Model(
            inputs  = model.inputs,
            outputs = [model.get_layer(last_conv_layer).output, model.output]
        )
        with tf.GradientTape() as tape:
            rgb_t = tf.cast(rgb_input[np.newaxis], tf.float32)
            dct_t = tf.cast(dct_input[np.newaxis], tf.float32)
            conv_out, preds = grad_model([rgb_t, dct_t])
            loss = preds[:, 0]

        grads   = tape.gradient(loss, conv_out)
        pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
        cam     = tf.reduce_sum(tf.multiply(pooled, conv_out[0]), axis=-1).numpy()
        cam     = np.maximum(cam, 0)
        cam     = cv2.resize(cam, IMG_SIZE)
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam
    except Exception:
        return None

def overlay_heatmap(original_rgb, heatmap, alpha=0.45):
    """Blend Grad-CAM heatmap over original image."""
    coloured = (cm.jet(heatmap)[:, :, :3] * 255).astype(np.uint8)
    base     = cv2.resize(original_rgb.astype(np.uint8), IMG_SIZE)
    blended  = cv2.addWeighted(base, 1 - alpha, coloured, alpha, 0)
    return blended

# ── DCT visualisation ─────────────────────────────────────────────────────────
def visualise_dct(dct_map):
    """Return a displayable version of the DCT frequency map."""
    vis = (dct_map[:, :, 0] * 255).astype(np.uint8)
    vis = cv2.applyColorMap(vis, cv2.COLORMAP_INFERNO)
    return cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.title("🔍 Deepfake Detector")
    st.caption("Phase 17 — Final Deployment")
    st.divider()

    st.markdown("**Model**")
    st.info("Dual-Stream Xception\nFF++ C23 adapted\nArtiFact zero-shot tested")

    st.divider()
    st.markdown("**Training pipeline**")
    st.markdown("""
- CIFAKE (diffusion objects)
- FaceForensics++ C23 (6 manipulation types)
- Zero-shot: ArtiFact (25 generators)
    """)

    st.divider()
    st.markdown("**Confidence threshold**")
    threshold = st.slider(
        "Decision boundary", 0.30, 0.70, 0.50, 0.01,
        help="Probability above this → FAKE. Default 0.50."
    )

    st.divider()
    st.markdown("**What this detects**")
    for gen, desc in GENERATOR_INFO.items():
        with st.expander(gen, expanded=False):
            st.caption(desc)

    st.divider()
    st.caption("Research: Beyond Binary Detection\nXAI Deepfake Localization via\nFrequency-Aware Segmentation")

# ── Main area ─────────────────────────────────────────────────────────────────
st.title("Beyond Binary Detection")
st.markdown("Upload an image to detect whether it is AI-generated and see **where** the model found evidence.")

uploaded = st.file_uploader(
    "Drop an image here (JPG, PNG, WEBP)",
    type=["jpg", "jpeg", "png", "webp"],
)

if uploaded is None:
    # Landing state
    col1, col2, col3 = st.columns(3)
    with col1:
        st.markdown("""
        #### 🧠 Dual-Stream Analysis
        The model processes every image through two parallel streams:
        - **RGB stream** — spatial textures, blending artefacts
        - **DCT stream** — frequency fingerprints of AI generation
        """)
    with col2:
        st.markdown("""
        #### 🗺️ Explainability
        Every prediction comes with a Grad-CAM heatmap showing **which pixels**
        pushed the model toward its decision — not just a score.
        """)
    with col3:
        st.markdown("""
        #### 📊 Evidence Panel
        See the raw DCT frequency map, confidence score,
        and probability breakdown alongside the verdict.
        """)
    st.stop()

# ── Run inference ─────────────────────────────────────────────────────────────
model = load_model()

pil_img   = Image.open(uploaded).convert("RGB")
img_array = np.array(pil_img)

rgb_input = preprocess_rgb(img_array)
dct_input = preprocess_dct(img_array)

with st.spinner("Analysing…"):
    prob = float(model.predict(
        [rgb_input[np.newaxis], dct_input[np.newaxis]], verbose=0
    )[0][0])

is_fake    = prob >= threshold
confidence = prob if is_fake else (1 - prob)
label      = "FAKE" if is_fake else "REAL"
uncertain  = CONFIDENCE_LOW < prob < CONFIDENCE_HIGH

# ── Layout ────────────────────────────────────────────────────────────────────
left, right = st.columns([1, 1], gap="large")

with left:
    st.markdown('<p class="section-header">Input Image</p>', unsafe_allow_html=True)
    st.image(pil_img, use_column_width=True)

with right:
    # Verdict card
    card_class = "verdict-fake" if is_fake else "verdict-real"
    icon       = "🔴" if is_fake else "🟢"
    st.markdown(
        f'<div class="{card_class}">'
        f'<h1 style="margin:0">{icon} {label}</h1>'
        f'<p style="font-size:1.4rem;margin:4px 0">'
        f'{"Fake" if is_fake else "Real"} probability: <b>{prob:.1%}</b></p>'
        f'{"<p style=\'color:#f39c12\'>⚠️ Low confidence — result is uncertain</p>" if uncertain else ""}'
        f'</div>',
        unsafe_allow_html=True
    )

    st.divider()

    # Probability bar
    st.markdown('<p class="section-header">Probability Breakdown</p>', unsafe_allow_html=True)
    p_col1, p_col2 = st.columns(2)
    with p_col1:
        st.metric("REAL probability", f"{(1-prob):.1%}")
    with p_col2:
        st.metric("FAKE probability", f"{prob:.1%}")

    st.progress(prob)

    st.divider()

    # Interpretation
    st.markdown('<p class="section-header">Interpretation</p>', unsafe_allow_html=True)
    if prob > 0.85:
        st.error("Strong AI generation signal detected in frequency domain.")
    elif prob > 0.65:
        st.warning("Moderate AI generation signal — likely manipulated.")
    elif prob > 0.35:
        st.info("Ambiguous signal — could be real or lightly processed.")
    else:
        st.success("No significant AI generation signal found.")

# ── Explainability panel ──────────────────────────────────────────────────────
st.divider()
st.markdown("### Explainability — Where Did the Model Look?")

with st.spinner("Generating Grad-CAM heatmap…"):
    heatmap = make_gradcam(model, rgb_input, dct_input)

e1, e2, e3 = st.columns(3)

with e1:
    st.markdown('<p class="section-header">Original (resized)</p>', unsafe_allow_html=True)
    display_rgb = cv2.resize(img_array, IMG_SIZE)
    st.image(display_rgb, use_column_width=True)
    st.caption("Input as seen by the RGB stream")

with e2:
    st.markdown('<p class="section-header">Grad-CAM Heatmap</p>', unsafe_allow_html=True)
    if heatmap is not None:
        blended = overlay_heatmap(img_array, heatmap)
        st.image(blended, use_column_width=True)
        st.caption("🔴 Red = high model attention  🔵 Blue = low attention")
    else:
        st.warning("Grad-CAM unavailable — layer name may differ in this model.")

with e3:
    st.markdown('<p class="section-header">DCT Frequency Map</p>', unsafe_allow_html=True)
    dct_vis = visualise_dct(dct_input)
    st.image(dct_vis, use_column_width=True)
    st.caption("Bright regions = high-frequency AI generation artifacts")

# ── Technical detail expander ─────────────────────────────────────────────────
with st.expander("Technical details", expanded=False):
    st.markdown(f"""
| Property | Value |
|---|---|
| Model architecture | Dual-Stream Xception |
| RGB input shape | 299 × 299 × 3 |
| DCT input shape | 299 × 299 × 3 (log-normalised) |
| Decision threshold | {threshold:.2f} |
| Raw FAKE probability | {prob:.6f} |
| Training datasets | CIFAKE, FaceForensics++ C23 |
| Zero-shot tested on | ArtiFact (25 generators) |
| Explainability method | Grad-CAM on last Xception conv layer |
    """)

'''

with open(f"{DEPLOY_DIR}/app.py", "w") as f:
    f.write(APP_CODE.strip())
print("✓ app.py written")

### requirements.txt

In [ ]:
REQUIREMENTS = """streamlit==1.35.0
tensorflow==2.16.1
opencv-python-headless==4.9.0.80
numpy==1.26.4
matplotlib==3.8.4
Pillow==10.3.0
huggingface-hub==0.23.2
"""

with open(f"{DEPLOY_DIR}/requirements.txt", "w") as f:
    f.write(REQUIREMENTS)
print("✓ requirements.txt written")

### Dockerfile

HuggingFace Spaces now requires the Docker SDK for Streamlit apps. This Dockerfile installs the OpenCV system dependencies that `opencv-python-headless` needs, copies the app, and starts Streamlit on port 7860 (the HF Spaces default).

In [ ]:
DOCKERFILE = """FROM python:3.10-slim

WORKDIR /app

RUN apt-get update && apt-get install -y \\
    libgl1-mesa-glx \\
    libglib2.0-0 \\
    libsm6 \\
    libxext6 \\
    libxrender-dev \\
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 7860

CMD [\"streamlit\", \"run\", \"app.py\",
     \"--server.port=7860\",
     \"--server.address=0.0.0.0\",
     \"--server.headless=true\",
     \"--browser.gatherUsageStats=false\"]
"""

with open(f"{DEPLOY_DIR}/Dockerfile", "w") as f:
    f.write(DOCKERFILE)
print("✓ Dockerfile written")

### Streamlit config + .gitignore

In [ ]:
CONFIG = """[theme]
base = \"dark\"
primaryColor = \"#4e8cff\"
backgroundColor = \"#0f1117\"
secondaryBackgroundColor = \"#1e2130\"
textColor = \"#e2e8f0\"
font = \"sans serif\"

[server]
maxUploadSize = 10
"""
with open(f"{DEPLOY_DIR}/.streamlit/config.toml", "w") as f:
    f.write(CONFIG)

GITIGNORE = """*.keras
*.h5
__pycache__/
.env
.streamlit/secrets.toml
*.pyc
.DS_Store
venv/
"""
with open(f"{DEPLOY_DIR}/.gitignore", "w") as f:
    f.write(GITIGNORE)

print("✓ .streamlit/config.toml written")
print("✓ .gitignore written")

### README.md

In [ ]:
README = open("/kaggle/working/deepfake-detector/README.md").read()
# README is already written by the packaging cell — just confirm
print("README lines:", len(README.splitlines()))
# If running fresh, write it:
README_CONTENT = '''
# deepfake-detector

**Beyond Binary Detection — XAI Deepfake Localization via Frequency-Aware Detection**

Live demo → [huggingface.co/spaces/YOUR_USERNAME/deepfake-detector](https://huggingface.co/spaces/YOUR_USERNAME/deepfake-detector)

---

## What This Is

A research prototype that detects AI-generated images and explains *where* in the image the evidence was found. It doesn't just say "Fake" — it shows you the pixels that gave it away using Grad-CAM heatmaps and DCT frequency maps.

Built across 17 phases as part of a BSc/MSc research project on explainable deepfake detection.

---

## How It Works

Every uploaded image goes through two parallel processing streams:

```
Image
  ├── RGB Stream    → Xception CNN → spatial texture features
  └── DCT Stream    → log-normalised frequency map → Xception CNN → frequency artefact features
                                ↓
                        Fusion + Dense head
                                ↓
                    REAL / FAKE  +  confidence score
                                ↓
                    Grad-CAM heatmap (where the model looked)
```

**RGB stream** catches: blending boundaries, skin texture inconsistencies, unnatural symmetry.
**DCT stream** catches: upsampling grid artefacts, generation frequency fingerprints, GAN checkerboard patterns.

---

## Training History

| Phase | Dataset | Purpose |
|---|---|---|
| Phases 1–10 | CIFAKE | Baseline — diffusion-generated objects |
| Phase 13 | CIFAKE | Backbone upgrade to Xception |
| Phase 14 | FaceForensics++ C23 | Domain adaptation — face-swap GANs |
| Phase 15 | ArtiFact + COCO2017 | Zero-shot generalization test (25 generators) |

The model was never trained on ArtiFact — Phase 15 is a pure zero-shot evaluation to test whether it learned general AI synthesis patterns, not dataset-specific memorisation.

---

## Generators the Model Has Seen

**During training (FF++ C23):**
- Deepfakes (autoencoder face swap)
- FaceSwap (3DMM-based)
- Face2Face (expression transfer)
- FaceShifter (high-fidelity swap)
- NeuralTextures (neural rendering)
- DeepFakeDetection (Google dataset)

**Zero-shot tested against (ArtiFact):**
StyleGAN1/2/3, Stable Diffusion, DDPM, Latent Diffusion, BigGAN, ProGAN, GANFormer, CIPS, CycleGAN, VQ-Diffusion, GLIDE, Palette, LAMA, MAT, and more.

---

## Repository Structure

```
deepfake-detector/
├── app.py                  ← Streamlit application (all UI + inference logic)
├── requirements.txt        ← Python dependencies
├── Dockerfile              ← For HuggingFace Spaces Docker deployment
├── .streamlit/
│   └── config.toml         ← Dark theme, upload size limit
├── .gitignore              ← Excludes model weights, secrets
└── README.md               ← This file
```

The model weights (`ffpp_adapted_final.keras`) are **not** in this repo. They are hosted separately on HuggingFace Hub and downloaded automatically at runtime by `huggingface_hub`.

---

## Deployment

### HuggingFace Spaces (live demo)

The app runs on HuggingFace Spaces using the Docker SDK. Any push to `main` triggers an automatic redeploy.

### Run locally

```bash
git clone https://github.com/YOUR_USERNAME/deepfake-detector.git
cd deepfake-detector

# Create virtual environment
python -m venv venv
source venv/bin/activate        # Windows: venv\Scripts\activate

# Install dependencies
pip install -r requirements.txt

# Place model weights in the project root (or update HF_REPO_ID in app.py)
# Then run:
streamlit run app.py
```

---

## Model Weights

Model weights are hosted on HuggingFace Hub:
`https://huggingface.co/YOUR_USERNAME/deepfake-detector-model`

To use a local copy, place `ffpp_adapted_final.keras` in the project root and update `HF_REPO_ID` in `app.py` to point to your repo, or change the load path to local.

---

## Results Summary

| Model | Dataset | AUC-ROC |
|---|---|---|
| Spatial CNN (EfficientNetB0) | CIFAKE | 0.46 |
| Frequency CNN (EfficientNetB0) | CIFAKE | 0.91 |
| Dual-Stream (EfficientNetB0) | CIFAKE | 0.93 |
| Dual-Stream Xception | CIFAKE | — (paste Phase 13 result) |
| FF++ Adapted Xception | FF++ C23 | 0.74 |
| **This model (zero-shot)** | **ArtiFact** | **— (paste Phase 15 result)** |

---

## Limitations

- Face-swap detection is strongest on frontal, well-lit faces (FF++ training distribution)
- Very heavily JPEG-compressed images may reduce frequency stream effectiveness
- The model was not trained on video — it analyses single frames only
- Highly novel generators not present in either training set may reduce confidence

---

## Research Context

This project is the practical implementation component of a research proposal investigating whether frequency-domain analysis + spatial CNNs can detect AI-generated images across unseen generators — and whether the detection can be made explainable through spatial heatmaps.

**Research questions answered:**
1. ✅ Does the dual-stream beat spatial-only? (AUC 0.93 vs 0.46)
2. ✅ Can heatmaps localise manipulation? (Grad-CAM + Integrated Gradients)
3. ✅ Does it fit in 15GB cloud storage? (Kaggle free tier throughout)

---

## Citation / Reference

If you use this work, please reference the research proposal:
> *Beyond Binary Detection: XAI Deepfake Localization via Frequency-Aware Segmentation*

'''
with open(f"{DEPLOY_DIR}/README.md", "w") as f:
    f.write(README_CONTENT.strip())
print("✓ README.md written")

## Step 2 — Upload Model Weights to HuggingFace Hub

The `.keras` model file is too large for GitHub (~200MB+). Host it on HuggingFace Hub as a model repository — it's free and `huggingface_hub` pulls it automatically at runtime.

Run the cell below **after downloading `ffpp_adapted_final.keras` from Phase 14**.

In [ ]:
# ── Upload model to HuggingFace Hub ──────────────────────────────────────
# Prerequisites:
#   1. pip install huggingface_hub
#   2. Create a free account at huggingface.co
#   3. Get your token from huggingface.co/settings/tokens (write access)
#   4. Create a new MODEL repo at huggingface.co/new (name: deepfake-detector-model)

# Uncomment and run once:
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_file(
#     path_or_fileobj="/kaggle/working/models/ffpp_adapted_final.keras",
#     path_in_repo="ffpp_adapted_final.keras",
#     repo_id="YOUR_HF_USERNAME/deepfake-detector-model",  # ← update this
#     repo_type="model",
#     token="hf_YOUR_TOKEN_HERE",                          # ← update this
# )
# print("Model uploaded to HuggingFace Hub")

print("Update HF_REPO_ID in app.py to match your HuggingFace username,")
print("then uncomment and run the upload block above.")

## Step 3 — Update Your Username in app.py

One line in `app.py` needs your HuggingFace username. Run this cell after uploading the model.

In [ ]:
HF_USERNAME = "YOUR_HF_USERNAME"  # ← paste your HuggingFace username here

app_path = f"{DEPLOY_DIR}/app.py"
with open(app_path) as f:
    content = f.read()

updated = content.replace(
    'HF_REPO_ID     = "YOUR_HF_USERNAME/deepfake-detector-model"',
    f'HF_REPO_ID     = "{HF_USERNAME}/deepfake-detector-model"'
)

with open(app_path, "w") as f:
    f.write(updated)

print(f"✓ HF_REPO_ID updated to: {HF_USERNAME}/deepfake-detector-model")

## Step 4 — Verify All Files

In [ ]:
import os

expected = [
    "app.py",
    "requirements.txt",
    "Dockerfile",
    "README.md",
    ".gitignore",
    ".streamlit/config.toml",
]

print("── Deployment package contents ─────────────────────")
all_ok = True
for fname in expected:
    fpath = os.path.join(DEPLOY_DIR, fname)
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) if exists else 0
    status = "✓" if exists else "✗"
    print(f"  {status}  {fname:<35} {size:>6} bytes")
    if not exists:
        all_ok = False

print()
print("All files present:" if all_ok else "MISSING FILES — re-run cells above")

## Step 5 — Package & Download

In [ ]:
import shutil

zip_path = "/kaggle/working/phase_17_deepfake_detector"
shutil.make_archive(zip_path, "zip", DEPLOY_DIR)

size_mb = os.path.getsize(zip_path + ".zip") / (1024 * 1024)
print(f"✓ Packaged: {zip_path}.zip  ({size_mb:.2f} MB)")
print()
print("Download this zip from Kaggle output panel, then:")
print("  1. Unzip it")
print("  2. git init && git remote add origin https://github.com/YOUR_USERNAME/deepfake-detector.git")
print("  3. git add . && git commit -m 'Initial deployment' && git push -u origin main")

## Full Deployment Instructions

Follow these steps in order after downloading the zip.

---

### A — GitHub Repository

**Repo name:** `deepfake-detector`

1. Go to [github.com/new](https://github.com/new)
2. Name: `deepfake-detector` | Visibility: Public | No template
3. Click **Create repository**
4. On your local machine:
```bash
unzip phase_17_deepfake_detector.zip -d deepfake-detector
cd deepfake-detector
git init
git add .
git commit -m "Phase 17: final deployment — Xception dual-stream deepfake detector"
git branch -M main
git remote add origin https://github.com/YOUR_USERNAME/deepfake-detector.git
git push -u origin main
```

---

### B — HuggingFace Model Repo (model weights)

1. Go to [huggingface.co/new](https://huggingface.co/new)
2. **Type:** Model | **Name:** `deepfake-detector-model` | Visibility: Public
3. Get your token: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → New token (Write)
4. Run the upload cell above (Step 2) with your username and token
5. Verify the file appears at: `huggingface.co/YOUR_USERNAME/deepfake-detector-model`

---

### C — HuggingFace Spaces (live app)

1. Go to [huggingface.co/new-space](https://huggingface.co/new-space)
2. **Name:** `deepfake-detector`
3. **SDK:** Docker ← important, not Streamlit
4. **Visibility:** Public
5. Click **Create Space**
6. In the Space, go to **Settings → Repository** and link your GitHub repo OR:
```bash
# Push directly to HF Spaces remote
git remote add space https://huggingface.co/spaces/YOUR_USERNAME/deepfake-detector
git push space main
```
7. HF Spaces builds the Docker container automatically — watch the **Build logs** tab
8. Build takes ~3–5 minutes. When done: 🟢 Running
9. Your app is live at: `https://huggingface.co/spaces/YOUR_USERNAME/deepfake-detector`

---

### D — Update README with live links

Once deployed, update these two lines in `README.md`:
```
Live demo → https://huggingface.co/spaces/YOUR_USERNAME/deepfake-detector
Model weights → https://huggingface.co/YOUR_USERNAME/deepfake-detector-model
```
Then `git commit -am 'Add live links' && git push`

## Troubleshooting

| Problem | Fix |
|---|---|
| Build fails: `libGL.so not found` | Already handled — Dockerfile installs `libgl1-mesa-glx` |
| `Model not found` error in app | Check `HF_REPO_ID` matches your HF username exactly |
| Grad-CAM returns `None` | The Xception layer name may differ — check `model.summary()` for the last conv layer name and update `last_conv_layer` in `make_gradcam()` |
| App loads but is slow | First load downloads the model — subsequent loads use cache. Free CPU tier is slow on first inference. |
| `git push` to HF Spaces times out | Use HTTPS not SSH: `https://huggingface.co/spaces/...` |
| Space shows `Building` forever | Check Build logs tab for errors — usually a missing dependency |

**Local test before pushing:**
```bash
# Place ffpp_adapted_final.keras in the project root
# Change HF_REPO_ID load to local path temporarily
streamlit run app.py
# Open http://localhost:8501
```

---
## Phase 17 Complete

| Deliverable | Location |
|---|---|
| Source code | `github.com/YOUR_USERNAME/deepfake-detector` |
| Model weights | `huggingface.co/YOUR_USERNAME/deepfake-detector-model` |
| Live demo | `huggingface.co/spaces/YOUR_USERNAME/deepfake-detector` |
| Download zip | `/kaggle/working/phase_17_deepfake_detector.zip` |

**All 17 phases complete. Update Phase 16 summary table to include Phase 17.**